# HLA-A*02:01 benchmark — Notebook 02: structure extraction and QC

**Scope:** Extract one HLA heavy chain / β2-microglobulin complex from deposited mmCIF structures, create a matched peptide-deleted model from **1HHI**, and report missing residues, engineered sequence differences and disulfide geometry. **This does not build AMBER topologies or equilibrate structures.**

**Inputs:** 1HHI (chains A/B/C), 6TDR and 6TDS (chains A/B). The deposited files may contain multiple copies; these chain IDs refer to *author* chain IDs and are verified before extraction. 6TDR/6TDS are engineered disulfide-constrained constructs and are *not* wild-type apo models.

**Terminology:** `WT_APO_deletion` denotes a *computational peptide-deletion starting structure*, not a validated equilibrium peptide-free HLA conformation.


In [1]:
!pip -q install biopython pandas numpy scipy matplotlib

In [2]:
from pathlib import Path
from urllib.request import urlretrieve
import copy, csv, json, hashlib
import numpy as np
import pandas as pd
from Bio.PDB import MMCIFParser, PDBIO, Select, Superimposer, Polypeptide
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.Align import PairwiseAligner

ROOT = Path('/content/HLA-A0201-Benchmark')
RAW = ROOT / '00_Experimental_PDB'
OUT = ROOT / '01_Structure_Preparation'
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
PDB_IDS = ['1HHI', '6TDR', '6TDS']
for pdb in PDB_IDS:
    f = RAW / pdb / f'{pdb}.cif'
    f.parent.mkdir(parents=True, exist_ok=True)
    if not f.exists():
        print('Downloading', pdb)
        urlretrieve(f'https://files.rcsb.org/download/{pdb}.cif', f)
    print(pdb, f, 'bytes=', f.stat().st_size, 'sha256=', hashlib.sha256(f.read_bytes()).hexdigest()[:16])


1HHI /content/HLA-A0201-Benchmark/00_Experimental_PDB/1HHI/1HHI.cif bytes= 640823 sha256= f2872a28c6156b3a
6TDR /content/HLA-A0201-Benchmark/00_Experimental_PDB/6TDR/6TDR.cif bytes= 848048 sha256= 07ec68ef0871c81e
6TDS /content/HLA-A0201-Benchmark/00_Experimental_PDB/6TDS/6TDS.cif bytes= 745547 sha256= 439e3221051f3840


## 1. Verify author chain identities and sequence annotations

We verify the author-chain mapping against `_entity_poly.pdbx_strand_id`, rather than assuming that Biopython's chain identifiers match mmCIF label-asym IDs.

In [3]:
for pdb in PDB_IDS:
    d = MMCIF2Dict(str(RAW / pdb / f'{pdb}.cif'))
    names = dict(zip(d.get('_entity.id',[]),d.get('_entity.pdbx_description',[])))
    print('\n',pdb)
    for ent, chains, seq in zip(d['_entity_poly.entity_id'], d['_entity_poly.pdbx_strand_id'], d['_entity_poly.pdbx_seq_one_letter_code_can']):
        print('entity', ent, 'author chains', chains, 'length', len(''.join(seq.split())), names.get(ent,''))



 1HHI
entity 1 author chains A,D length 275 CLASS I HISTOCOMPATIBILITY ANTIGEN (HLA-A*0201) (ALPHA CHAIN)
entity 2 author chains B,E length 100 BETA 2-MICROGLOBULIN
entity 3 author chains C,F length 9 INFLUENZA A MATRIX PROTEIN M1 (RESIDUES 58-66)

 6TDR
entity 1 author chains A,C length 276 MHC class I antigen
entity 2 author chains B,D length 100 Beta-2-microglobulin

 6TDS
entity 1 author chains A,C length 276 MHC class I antigen
entity 2 author chains B,D length 100 Beta-2-microglobulin


## 2. Extract the biological subcomplex

For 1HHI, A = heavy chain, B = β2m and C = M1 peptide. `WT_APO_deletion` is the same coordinate model with chain C removed. For 6TDR/6TDS, use chains A and B. Water and non-polymer crystallization components are excluded here, but must be reconsidered during AMBER preparation.

In [4]:
parser=MMCIFParser(QUIET=True, auth_chains=True, auth_residues=True)
class OnlyChains(Select):
    def __init__(self, keep): self.keep=set(keep)
    def accept_model(self, model): return model.id == 0
    def accept_chain(self, chain): return chain.id in self.keep
    def accept_residue(self, residue): return residue.id[0] == ' '
    def accept_atom(self, atom):
        return atom.get_altloc() in (' ', 'A')

def save_chains(structure, chains, path):
    available=set(structure[0].child_dict)
    if not set(chains).issubset(available):
        raise ValueError(f'Missing requested chains {set(chains)-available}; found {sorted(available)}')
    io=PDBIO(); io.set_structure(structure)
    io.save(str(path), select=OnlyChains(chains))
    if path.stat().st_size < 100: raise RuntimeError('Export appears empty')

sources={p:parser.get_structure(p,str(RAW/p/f'{p}.cif')) for p in PDB_IDS}
targets={
  'WT_BOUND_1HHI':('1HHI',['A','B','C']),
  'WT_APO_deletion_1HHI':('1HHI',['A','B']),
  'DS_APO_6TDR':('6TDR',['A','B']),
  'DS_APO_6TDS':('6TDS',['A','B'])
}
paths={}
for name,(pdb,ch) in targets.items():
    folder=OUT/name; folder.mkdir(exist_ok=True)
    path=folder/f'{name}.pdb'
    save_chains(sources[pdb],ch,path); paths[name]=path
    print(name, '→', path, f'({path.stat().st_size/1024:.1f} KiB)')


WT_BOUND_1HHI → /content/HLA-A0201-Benchmark/01_Structure_Preparation/WT_BOUND_1HHI/WT_BOUND_1HHI.pdb (249.7 KiB)
WT_APO_deletion_1HHI → /content/HLA-A0201-Benchmark/01_Structure_Preparation/WT_APO_deletion_1HHI/WT_APO_deletion_1HHI.pdb (244.1 KiB)
DS_APO_6TDR → /content/HLA-A0201-Benchmark/01_Structure_Preparation/DS_APO_6TDR/DS_APO_6TDR.pdb (243.9 KiB)
DS_APO_6TDS → /content/HLA-A0201-Benchmark/01_Structure_Preparation/DS_APO_6TDS/DS_APO_6TDS.pdb (243.8 KiB)


## 3. Validate peptide deletion and structural identity

A matched pair should contain exactly the same protein heavy-chain and β2m coordinates before peptide deletion. The peptide sequence and count are checked explicitly.

In [7]:
from Bio.SeqUtils import seq1
from Bio.PDB import PDBParser

def protein_residues(chain):
    return [r for r in chain if r.id[0]==' ' and seq1(r.resname, custom_map={'MSE':'M'})!='X']
def seq_of(chain):
    return ''.join(seq1(r.resname,custom_map={'MSE':'M'}) for r in protein_residues(chain))

pdb_parser = PDBParser(QUIET=True)
bound=pdb_parser.get_structure('bound',str(paths['WT_BOUND_1HHI']))
apo=pdb_parser.get_structure('apo',str(paths['WT_APO_deletion_1HHI']))
print('Bound chains:', [(c.id,len(protein_residues(c)), seq_of(c)[:16]) for c in bound[0]])
print('Apo chains:', [(c.id,len(protein_residues(c)), seq_of(c)[:16]) for c in apo[0]])
assert seq_of(bound[0]['C'])=='GILGFVFTL', 'Unexpected peptide sequence!'
assert 'C' not in apo[0]
for ch in ['A','B']:
    b={a.get_full_id()[-2:]:a.coord for a in bound[0][ch].get_atoms()}
    a={at.get_full_id()[-2:]:at.coord for at in apo[0][ch].get_atoms()}
    assert b.keys()==a.keys()
    print(ch,'max coordinate difference:', max(float(np.linalg.norm(b[k]-a[k])) for k in a),'Å')

Bound chains: [('A', 275, 'GSHSMRYFFTSVSRPG'), ('B', 100, 'MIQRTPKIQVYSRHPA'), ('C', 9, 'GILGFVFTL')]
Apo chains: [('A', 275, 'GSHSMRYFFTSVSRPG'), ('B', 100, 'MIQRTPKIQVYSRHPA')]
A max coordinate difference: 0.0 Å
B max coordinate difference: 0.0 Å


## 4. Sequence and disulfide quality checks

This is an **inventory**: it does not repair gaps. Cysteines are numbered as in exported author residue IDs; crosslinks must be validated in `tleap` (PDB exports do not by themselves guarantee bond topology).

In [9]:
from scipy.spatial.distance import cdist
from Bio.PDB import PDBParser

pdb_parser = PDBParser(QUIET=True)
rows=[]; ssrows=[]
for name,path in paths.items():
    st=pdb_parser.get_structure(name,str(path)); model=st[0]
    for ch in model:
        res=protein_residues(ch)
        nums=[r.id[1] for r in res]
        gaps=[(x,y) for x,y in zip(nums,nums[1:]) if y-x>1]
        missing_bb={r.id[1]:[v for v in ['N','CA','C','O'] if v not in r] for r in res}
        missing_bb={i:v for i,v in missing_bb.items() if v}
        rows.append(dict(system=name,chain=ch.id,residue_count=len(res),first_residue=nums[0] if nums else None,last_residue=nums[-1] if nums else None,numbering_gaps=str(gaps[:20]),missing_backbone=str(missing_bb),sequence=seq_of(ch)))
    cysteines=[]
    for ch in model:
        for res in ch:
            if res.resname in ('CYS','CYX') and 'SG' in res:
                cysteines.append((ch.id,res.id[1],res['SG'].coord))
    for i in range(len(cysteines)):
        for j in range(i+1,len(cysteines)):
            a,b=cysteines[i],cysteines[j]; dist=float(np.linalg.norm(a[2]-b[2]))
            if dist < 3.0:
                ssrows.append(dict(system=name,cys1=f'{a[0]}:{a[1]}',cys2=f'{b[0]}:{b[1]}',SG_distance_A=round(dist,3)))
qcdf=pd.DataFrame(rows)
ssdf=pd.DataFrame(ssrows)
display(qcdf.drop(columns='sequence'))
print('Geometrically close S–S candidates (not automatic bond assignments):')
display(ssdf)
qcdf.to_csv(OUT/'structure_qc.csv',index=False)
ssdf.to_csv(OUT/'disulfide_candidates.csv',index=False)

,system,chain,residue_count,first_residue,last_residue,numbering_gaps,missing_backbone
0,WT_BOUND_1HHI,A,275,1,275,[],{}
1,WT_BOUND_1HHI,B,100,0,99,[],{}
2,WT_BOUND_1HHI,C,9,1,9,[],{}
3,WT_APO_deletion_1HHI,A,275,1,275,[],{}
4,WT_APO_deletion_1HHI,B,100,0,99,[],{}
5,DS_APO_6TDR,A,275,1,275,[],{}
6,DS_APO_6TDR,B,100,1,100,[],{}
7,DS_APO_6TDS,A,275,1,275,[],{}
8,DS_APO_6TDS,B,100,1,100,[],{}


Geometrically close S–S candidates (not automatic bond assignments):


,system,cys1,cys2,SG_distance_A
0,WT_BOUND_1HHI,A:101,A:164,2.029
1,WT_BOUND_1HHI,A:203,A:259,2.027
2,WT_BOUND_1HHI,B:25,B:80,2.021
3,WT_APO_deletion_1HHI,A:101,A:164,2.029
4,WT_APO_deletion_1HHI,A:203,A:259,2.027
5,WT_APO_deletion_1HHI,B:25,B:80,2.021
6,DS_APO_6TDR,A:84,A:139,2.060
7,DS_APO_6TDR,A:101,A:164,2.163
8,DS_APO_6TDR,A:203,A:259,1.959
9,DS_APO_6TDR,B:26,B:81,2.046


## 5. Structural comparison of the peptide-binding groove

Align deposited heavy-chain Cα coordinates using sequence correspondences. This is a global heavy-chain diagnostic only; subsequent work will define groove-specific alignments and pocket coordinates.

In [11]:
from Bio.Align import PairwiseAligner
from Bio.PDB import PDBParser

pdb_parser = PDBParser(QUIET=True)
aligner=PairwiseAligner(); aligner.mode='global'; aligner.match_score=2;aligner.mismatch_score=-1;aligner.open_gap_score=-8;aligner.extend_gap_score=-0.5
ref=pdb_parser.get_structure('ref',str(paths['WT_BOUND_1HHI']))[0]['A']; rseq=seq_of(ref); rres=protein_residues(ref)
summary=[]
for name in ['WT_APO_deletion_1HHI','DS_APO_6TDR','DS_APO_6TDS']:
    comp=pdb_parser.get_structure('comp',str(paths[name]))[0]['A']; cseq=seq_of(comp); cres=protein_residues(comp)
    a=aligner.align(rseq,cseq)[0]
    fixed=[];moving=[]; substitutions=[]
    for (rs,re),(cs,ce) in zip(a.aligned[0],a.aligned[1]):
        for ri,ci in zip(range(rs,re),range(cs,ce)):
            if rseq[ri]!=cseq[ci]:substitutions.append((rres[ri].id[1],rseq[ri],cres[ci].id[1],cseq[ci]))
            if 'CA' in rres[ri] and 'CA' in cres[ci]:
                fixed.append(rres[ri]['CA']);moving.append(cres[ci]['CA'])
    sup=Superimposer();sup.set_atoms(fixed,moving)
    summary.append(dict(system=name,matched_CA=len(fixed),heavy_chain_CA_RMSD_A=round(sup.rms,3),substitutions=str(substitutions)))
display(pd.DataFrame(summary))
pd.DataFrame(summary).to_csv(OUT/'heavy_chain_alignment.csv',index=False)

,system,matched_CA,heavy_chain_CA_RMSD_A,substitutions
0,WT_APO_deletion_1HHI,275,0.000,[]
1,DS_APO_6TDR,275,0.939,"[(84, 'Y', 84, 'C'), (139, 'A', 139, 'C'), (24..."
2,DS_APO_6TDS,275,0.902,"[(84, 'Y', 84, 'C'), (139, 'A', 139, 'C'), (24..."


## 6. Package outputs for GitHub and subsequent AMBER preparation

These files contain extracted **starting coordinates**, not validated equilibrated systems. Inspect `structure_qc.csv` and disulfide candidates before assigning topologies. The missing-residue report cannot distinguish missing terminal segments from construct design; use the original mmCIF sequence mapping for that assessment.

In [12]:
import zipfile
archive=Path('/content/HLA_Step2_Preparation_QC.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,arcname=str(p.relative_to(ROOT)))
print('Created:',archive, 'size:',archive.stat().st_size)
from google.colab import files
files.download(str(archive))


Created: /content/HLA_Step2_Preparation_QC.zip size: 253958


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Decision gate before Step 3

- Verify all extracted chains and sequences.
- Inspect unresolved/alternate atoms and missing residues against the experimental mmCIF files.
- Confirm engineered disulfides from `_struct_conn` and actual SG geometry.
- Decide whether to simulate the peptide-deletion WT apo model, a structurally reconstructed apo model, or both.
- Do **not** run standard AMBER preparation on structures with unresolved gaps or chemically inconsistent crosslinks.

**Next notebook:** `03_HLA_AMBER_Preparation.ipynb` for a documented AMBER22 topology, explicit solvent and initial restrained equilibration.